# 05 — Live objects: `HostBase` and `opts`

In the previous tutorials, `Nematics3D` objects did more than store data. A visual object could be modified after it had already been drawn, and the existing figure changed with it.

That behavior reflects a common design used by many `Nematics3D` objects: **configuration is attached to a live object, and changing a managed input can update dependent state.**

`HostBase` provides this reactive object pattern. Most users do not create a `HostBase` directly. What matters is recognizing the pattern when a concrete object uses it.


## Return to an object we have already seen

We will use a director visualization created from the same $Q$-tensor example. This keeps the discussion connected to the objects from Tutorials 01–04 rather than introducing a synthetic class.


In [ ]:
import numpy as np
import nematics3d as n3d

Q = np.load("../../example/data/Q_example_workflow.npy")
Q_obj, figure = n3d.quick_visualize_q(Q=Q)
n_plane = Q_obj.objs["n-plane"]
visual = n_plane.visual_nb
visual


## First ask: what can I change?

Tutorial 03 introduced `show_readable_attrs()`. For a live host object there is a second question that is often even more useful: **which values are intended to be changed?**


In [ ]:
visual.show_modifiable_attrs()


## `opts`: configuration that belongs to the object

Many `HostBase` descendants have a paired `opts` object. It collects configuration values that control how the host behaves.

For a visual object, these can include properties such as glyph size or appearance. The exact fields depend on the concrete class, so the useful habit is to inspect the options rather than memorize a universal list.


In [ ]:
visual.opts


## The important part: the options are live

An attached `opts` object is not just a passive record of settings. Once it belongs to a host, changing a public option is routed back through that host's update machinery. Dependent state can therefore be recomputed immediately.

This is the mechanism behind the interactive editing we saw in Tutorial 02: the control panel and Python interface are changing the same live object model.

For example, after identifying a modifiable size option with `show_modifiable_attrs()`, changing it through `visual.opts` updates the existing visualization rather than requiring us to rebuild the whole $Q$-field workflow.


## `act_commit()`: update several inputs as one change

Direct assignment is convenient for one setting. When several inputs belong to one logical update, `HostBase` also provides `act_commit(...)`.

Conceptually:

```python
obj.act_commit(option_a=..., option_b=...)
```

The concrete object decides what those options mean. `HostBase` supplies the common update route so validation, option application, recomputation, and synchronization can happen in a controlled pipeline.

You do not need `act_commit()` for every edit. The important distinction is:

- simple public assignment is convenient for a single live change;
- `act_commit(...)` is the explicit interface for a coordinated or batched change.


## A useful mental model

A `HostBase` descendant can be thought of as a small managed system:

```text
host inputs + opts
        ↓
   managed update
        ↓
calculated state / generated objects / visualization
```

This also explains the naming vocabulary from Tutorial 03. `raw_...` and `state_...` commonly describe inputs, `opts` contains configuration inputs, while `calc_...` and `entity_...` commonly expose results produced from those inputs.


## What you do not need yet

`HostBase` also supports option snapshots, protected fields, wrappers, synchronization callbacks, and other coordination mechanisms. Those features matter when building or composing more complicated `Nematics3D` objects, but they are not required for the basic user workflow. The `HostBase` reference tutorial documents them when you need them.


## The main idea

You normally interact with concrete scientific and visual objects, not with `HostBase` itself. When such an object exposes `opts`, treat those options as **live configuration**:

1. use `show_modifiable_attrs()` to discover the intended write surface;
2. inspect `obj.opts` to understand its configuration;
3. change a public option for a simple live update;
4. use `act_commit(...)` when several inputs should be applied as one controlled update.

The result is the design principle we have already been using since Tutorial 02: **change the object you already have instead of rebuilding the entire analysis from scratch.**
